# 🧪 Lab — Can You Trust This Dataset?

## Overview

You have been given a dataset containing e-commerce sales data.

Your manager has already used this dataset to report revenue metrics to leadership.

Your job is to determine:

> **Can this dataset be trusted for analysis?**

---

## IMPORTANT

To get the dataset, navigate to the `./data/raw/` folder. 

You should see a Python file called: `generate_sales_data.py`.

**IN YOUR TERMINAL** inside the `./data/raw/` folder run the following command:

```
python3 generate_sales_data.py
```

You should then have a `sales.csv` file in the folder after it runs.

In [2]:
# Run your imports here`
import os
from pathlib import Path
import pandas as pd
import numpy as np
RAW = Path("..\data") / "raw"
PROCESSED = Path("..\data") / "processed"

Load the dataset.

Name the dataframe `sales`

In [3]:
sales = pd.read_csv(RAW/"sales.csv")

sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24


---

### Initial Analysis

In [4]:
sales.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         305 non-null    int64  
 1   customer_id      293 non-null    float64
 2   product_id       305 non-null    int64  
 3   revenue          305 non-null    object 
 4   quantity         305 non-null    int64  
 5   unit_price       305 non-null    float64
 6   status           305 non-null    object 
 7   payment_method   305 non-null    object 
 8    region          305 non-null    object 
 9   order_date       305 non-null    object 
dtypes: float64(2), int64(3), object(5)
memory usage: 24.0+ KB


In [5]:
# TODO: Calculate total revenue

# first fix revenue (remove $ and turn into float)
sales['revenue']=sales['revenue'].str.replace('$','')
sales['revenue']=sales['revenue'].astype(float)

In [6]:
# sales.info()
total_revenue=sales['revenue'].sum()
print("total revenue = $ ",round(total_revenue,2))

# TODO: Calculate average order value
ave_revenue=total_revenue/len(sales['revenue'])
print(f"average order revenue = $ ",round(ave_revenue,2))

total revenue = $  81819.03
average order revenue = $  268.26


In [7]:
# clean up the status column
# print(sales['status'].value_counts())
sales["status"] = sales["status"].str.title().str.strip()
sales['status'] = sales['status'].str.replace('Completed', 'Complete').str.replace('Returned', 'Return')
# print(sales['status_clean'].value_counts())

In [8]:
# TODO: Calculate revenue by status

# sales.groupby("status")
sales.groupby("status")["revenue"].sum()


status
Cancelled    25583.66
Complete     48393.16
Return        7134.60
Unknown        707.61
Name: revenue, dtype: float64

---

### Data Inspection

Call the usual Pandas methods to inspect a dataset.

In [9]:
# Display first 5 rows
# sales.head(5)
# Print shape and column names
print("matrix size:", sales.shape)
# print("Header names:", sales.columns.tolist()) 
#header names have white spaces that need cleaning
sales.columns = (sales.columns.str.strip().str.lower())
header_names=sales.columns.tolist()
print("Header names:", header_names) 
sales.describe()

matrix size: (305, 10)
Header names: ['order_id', 'customer_id', 'product_id', 'revenue', 'quantity', 'unit_price', 'status', 'payment_method', 'region', 'order_date']


,order_id,customer_id,product_id,revenue,quantity,unit_price
count,305.000000,293.000000,305.000000,305.000000,305.000000,305.000000
mean,1149.363934,38.709898,343.629508,268.259115,2.819672,49.615607
std,86.692792,23.068268,1114.916428,590.764366,1.458958,20.288912
min,1000.000000,-1.000000,200.000000,-300.910000,0.000000,-96.570000
25%,1075.000000,20.000000,208.000000,140.710000,2.000000,40.240000
50%,1149.000000,38.000000,216.000000,201.840000,3.000000,49.600000
75%,1224.000000,60.000000,223.000000,256.770000,4.000000,60.170000
max,1299.000000,79.000000,9999.000000,6178.250000,5.000000,97.510000


---

### Idenfity Issues

Feel free to break this up across multiple cells

In [10]:
# TODO: Check for missing values
# sales.isna().sum()   # 12 customer Ids are n/a

In [11]:
# TODO: Check for duplicates
# sales.duplicated().sum()  # 4 duplicates found

In [12]:
# TODO: Check for invalid revenue
# (sales["revenue"]<0).sum() # 8 negative values

In [13]:
# TODO: Check for invalid quantities
(sales["quantity"]<=0).sum() # 6 less than or equal to zero quantities

np.int64(6)

In [14]:
# check for outliers
def outliers(column):
    q1=column.quantile(0.25)
    q3 =column.quantile(0.75)
    print(q1, q3)
#calculate interquartile range
    iqr=q3-q1
    lower_bound=q1-1.5*iqr
    upper_bound=q3+1.5*iqr
    print(iqr, lower_bound, upper_bound)
    return (lower_bound, upper_bound)


In [15]:
lower_bound, upper_bound = outliers(sales["revenue"])
revenue_outliers = sales[
    (sales["revenue"] < lower_bound) |
    (sales["revenue"] > upper_bound)]

print(len(revenue_outliers))

140.71 256.77
116.05999999999997 -33.37999999999997 430.85999999999996
17


In [16]:
lower, upper = outliers(sales["unit_price"])
unit_price_outliers = sales[
    (sales["unit_price"] < lower) |
    (sales["unit_price"] > upper)]
print(len(unit_price_outliers))

40.24 60.17
19.93 10.345000000000002 90.065
7


In [17]:
# TODO: Check for inconsistent categories
# sales["status_clean"].value_counts()   
sales["payment_method"].value_counts()  # remove space after payment_method, and all kinds of stuff.
# sales["region"].value_counts() # this is a mess as well

payment_method
Gift Card        86
Credit Card      73
Debit Card       66
PayPal           58
paypal            5
Unknown           4
giftcard          4
credit card       3
CreditCard        3
Pay Pal           2
 CREDIT CARD      1
Name: count, dtype: int64

In [18]:
# TODO: Check for invalid dates
sales["order_date"] = pd.to_datetime(sales["order_date"], format="%Y-%m-%d", errors="coerce")
sales['order_date'].isna().sum()   #all dates good


np.int64(0)

---

### Create Validation Columns

In [19]:
# TODO: Create validation flags
sales["duplicate_order_id"] = sales["order_id"].duplicated(keep=False)
sales['missing_order_id']=sales['order_id'].isna()
sales['missing_customer_id']=sales['customer_id'].isna()
sales['missing_product_id']=sales['product_id'].isna()
sales["negative_revenue"] = sales["revenue"] < 0 | (sales["revenue"] < lower_bound) | (sales["revenue"] > upper_bound)
sales["invalid_quantity"] = sales["quantity"] <= 0
sales["invalid_unit_price"] = sales["unit_price"] <= 0 | (sales["unit_price"] < lower) | (sales["unit_price"] > upper)
today = pd.Timestamp.today()
sales["invalid_date"] = sales["order_date"]>today
sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,duplicate_order_id,missing_order_id,missing_customer_id,missing_product_id,negative_revenue,invalid_quantity,invalid_unit_price,invalid_date
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26,False,False,False,False,False,False,False,False
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07,False,False,False,False,False,True,False,False
2,1002,72.0,209,286.86,1,65.15,Return,Credit Card,West,2023-11-26,False,False,False,False,False,False,False,False
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08,False,False,False,False,False,False,False,False
4,1004,21.0,225,246.81,1,33.59,Return,Gift Card,West,2023-05-24,False,False,False,False,False,False,False,False


---

### Summarize Issues



In [20]:
# TODO: Summarize validation columns
validation_columns = [
    "duplicate_order_id",
    "missing_order_id",
    "missing_customer_id",
    "missing_product_id",
    "negative_revenue",
    "invalid_quantity",
    "invalid_unit_price",
    "invalid_date"
]
# print(validation_columns)

sales[validation_columns].sum()

duplicate_order_id     33
missing_order_id        0
missing_customer_id    12
missing_product_id      0
negative_revenue        8
invalid_quantity        6
invalid_unit_price      6
invalid_date            6
dtype: int64

In [21]:
#take a look at the problem rows
sales["has_quality_issue"] = sales[validation_columns].any(axis=1)
problem_rows = sales[sales['has_quality_issue']]
print("problem row count:",len(problem_rows))
problem_rows.head()

problem row count: 70


,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,duplicate_order_id,missing_order_id,missing_customer_id,missing_product_id,negative_revenue,invalid_quantity,invalid_unit_price,invalid_date,has_quality_issue
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07,False,False,False,False,False,True,False,False,True
12,1012,30.0,222,138.09,5,46.01,Cancelled,Credit Card,East,2023-05-18,True,False,False,False,False,False,False,False,True
13,1012,38.0,201,175.90,1,40.03,Return,Gift Card,Midwest,2035-01-01,True,False,False,False,False,False,False,True,True
36,1036,NaN,225,238.95,3,57.03,Complete,Credit Card,east,2023-03-07,False,False,True,False,False,False,False,False,True
38,1038,73.0,224,191.84,1,31.26,Complete,Unknown,Midwest,2023-09-11,True,False,False,False,False,False,False,False,True


---

### Assertions

In [22]:
# TODO: Write assertions
def validate_sales_data(df):
    assert df["order_id"].is_unique, "Order ID must be unique"
    assert df["order_id"].notna().all(), "Customer ID cannot be missing"
    assert df["customer_id"].notna().all(), "Customer ID cannot be missing"
    assert df["product_id"].notna().all(), "Product ID cannot be missing"
    assert (df['revenue'] >= 0).all(), "Revenue cannot be negative"
    assert (df["quantity"] > 0).all(), "Quantity must be greater than zero"
    assert (df["unit_price"] > 0).all(), "Unit Price must be greater than zero"

    # valid_status = ["Complete", "Cancelled", "Return", "Unknown"]
    # assert df.isin(valid_statuses).all(), "Status contains an invalid status"
    # valid_payment = ["unknown", "debitcard", "paypall", "giftcard", "creditcard"]
    # assert df.isin(valid_payment).all(), "Payment menthod contains an invalid method"
    # valid_region = ["north", "east","south", "west", "midwest", "northeast"]
    # assert df.isin(valid_region).all(), "Region contains an invalid location"

    return True

---

### Now clean your dataset

Save it as `clean_sales`

In [23]:
# clean up the Payment method column
# print(sales['payment_method'].value_counts())
sales["payment_method_clean"] = sales["payment_method"].str.lower().str.strip().str.replace(" ", "", regex=False)
print(sales['payment_method_clean'].value_counts())

# clean up the region column
# print(sales['region'].value_counts())
sales["region_clean"] = sales["region"].str.lower().str.strip().str.replace("-", "", regex=False)
print(sales['region_clean'].value_counts())
sales.head()

payment_method_clean
giftcard      90
creditcard    80
debitcard     66
paypal        65
unknown        4
Name: count, dtype: int64
region_clean
east         80
west         80
midwest      76
south        63
northeast     6
Name: count, dtype: int64


,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,...,missing_order_id,missing_customer_id,missing_product_id,negative_revenue,invalid_quantity,invalid_unit_price,invalid_date,has_quality_issue,payment_method_clean,region_clean
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26,...,False,False,False,False,False,False,False,False,unknown,east
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07,...,False,False,False,False,True,False,False,True,debitcard,west
2,1002,72.0,209,286.86,1,65.15,Return,Credit Card,West,2023-11-26,...,False,False,False,False,False,False,False,False,creditcard,west
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08,...,False,False,False,False,False,False,False,False,creditcard,east
4,1004,21.0,225,246.81,1,33.59,Return,Gift Card,West,2023-05-24,...,False,False,False,False,False,False,False,False,giftcard,west


In [24]:
# TODO: Filter out problematic rows
clean_sales = sales[~sales["has_quality_issue"]].copy()
# clean_sales.head()
clean_sales.describe()

,order_id,customer_id,product_id,revenue,quantity,unit_price,order_date
count,235.000000,235.000000,235.000000,235.000000,235.000000,235.000000,235
mean,1146.289362,38.765957,340.506383,284.709574,2.851064,51.131872,2023-07-01 00:06:07.659574528
min,1000.000000,-1.000000,200.000000,14.630000,1.000000,10.850000,2023-01-01 00:00:00
25%,1074.500000,20.000000,209.000000,148.365000,2.000000,40.625000,2023-04-05 00:00:00
50%,1145.000000,38.000000,216.000000,205.560000,3.000000,49.280000,2023-07-02 00:00:00
75%,1222.500000,60.000000,223.500000,258.590000,4.000000,61.655000,2023-09-27 12:00:00
max,1297.000000,79.000000,9999.000000,6178.250000,5.000000,97.510000,2023-12-30 00:00:00
std,85.592014,23.429119,1100.690799,621.050432,1.401750,16.238753,NaN


In [25]:
clean_sales['payment_method']= clean_sales['payment_method_clean']
clean_sales['region']= clean_sales['region_clean']
clean_sales.head()
clean_sales=clean_sales[header_names]
clean_sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
0,1000,52.0,217,124.55,3,52.18,Complete,unknown,east,2023-02-26
2,1002,72.0,209,286.86,1,65.15,Return,creditcard,west,2023-11-26
3,1003,61.0,221,259.37,3,47.06,Complete,creditcard,east,2023-04-08
4,1004,21.0,225,246.81,1,33.59,Return,giftcard,west,2023-05-24
5,1005,75.0,202,6178.25,5,14.43,Cancelled,giftcard,west,2023-05-05


In [26]:
#sales_clean has all cleaned 

validate_sales_data(clean_sales)

True

In [30]:
clean_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 235 entries, 0 to 297
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        235 non-null    int64         
 1   customer_id     235 non-null    float64       
 2   product_id      235 non-null    int64         
 3   revenue         235 non-null    float64       
 4   quantity        235 non-null    int64         
 5   unit_price      235 non-null    float64       
 6   status          235 non-null    object        
 7   payment_method  235 non-null    object        
 8   region          235 non-null    object        
 9   order_date      235 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(3), object(3)
memory usage: 20.2+ KB


---

### Re-run the analysis

Reference the lab `README` to remember which metrics to investigate.

In [28]:
# TODO: Recalculate metrics
# clean_sales.info()
total_revenue_clean=clean_sales['revenue'].sum()
print("cleaned total revenue = $ ",round(total_revenue_clean,2))

# TODO: Calculate average order value
ave_revenue_clean=total_revenue_clean/len(clean_sales['revenue'])
print(f"cleaned average order revenue = $ ",round(ave_revenue_clean,2))

clean_sales.groupby("status")["revenue"].sum()

cleaned total revenue = $  66906.75
cleaned average order revenue = $  284.71


status
Cancelled    20262.47
Complete     40061.33
Return        5875.34
Unknown        707.61
Name: revenue, dtype: float64

---

### Data Quality Report

In [29]:
sales[validation_columns].sum()


duplicate_order_id     33
missing_order_id        0
missing_customer_id    12
missing_product_id      0
negative_revenue        8
invalid_quantity        6
invalid_unit_price      6
invalid_date            6
dtype: int64

---

## Export cleaned dataset and report to the appropriate folder

In [31]:
clean_sales.to_csv(PROCESSED/"clean_sales.csv", index=False)

---

### Reflection Questions.

Answer the reflection questions from the README below in markdown cells.

### 
🔹 Part 1 — Initial Analysis
 Do you trust these results? Why or why not?
 
 NO, i don't trust the results based on the revenue is NOT equal to the unit price * quantity
 otherwise, besides a few outliers and negatives, it didn't look too bad.
 
### * Identify Data Issues and summarize findings

duplicate_order_id     33

missing_order_id        0

missing_customer_id    12

missing_product_id      0

negative_revenue        8

invalid_quantity        6

invalid_unit_price      6

invalid_date            6


### 🔹 Part 5 — Assertions
my assertions ran just fine once i got rid of all the issues, otherwise it stopped on the first issue it came to.

### 🔹 Part 6 — Create a Clean Dataset
- What did you remove?
- I removed all the problem data sets.  i considered recalculating the revenue based on the price and quantity, but since they were all independant, i didn't know what the TRUE numbers should be

I also deleted any of the outliers above and below the limits for unit price and revenue.  again, i didn't know what the real numbers should be.  maybe i should have not deleted the revenue below zero, but made it zero?  i was not familiar with this data types to make that decision.

I also cleaned up the three columns with categories, fixed the spaces, dashes, upper and lower cases
I cleaned up the headers. there were some white spaces, so i took care of those right away.

### 🔹 Part 7 — Re-run Analysis
Compare results before and after cleaning.
total revenue = $  81819.03
average order revenue = $  268.26
status
Cancelled    25583.66
Complete     48393.16
Return        7134.60
Unknown        707.61

cleaned total revenue = $  66906.75
cleaned average order revenue = $  284.71
status
Cancelled    20262.47
Complete     40061.33
Return        5875.34
Unknown        707.61

> What changed?

total revenue down by almost 20% - thats huge!
revenue down on each catagory except the unknown, i guess there wasn't any problems with that category.

### 🔹 Part 8 — Data Quality Report

Create a data-quality report that includes:

- Counts of each issue  
- Summary statistics  
- Category breakdowns
  
duplicate_order_id     33

missing_order_id        0

missing_customer_id    12

missing_product_id      0

negative_revenue        8

invalid_quantity        6

invalid_unit_price      6

invalid_date            6

	order_id	customer_id	product_id	revenue	quantity	unit_price	order_date
    
count	235.000000	235.000000	235.000000	235.000000	235.000000	235.000000	235

mean	1146.289362	38.765957	340.506383	284.709574	2.851064	51.131872	2023-07-01 00:06:07.659574528

min	1000.000000	-1.000000	200.000000	14.630000	1.000000	10.850000	2023-01-01 00:00:00

25%	1074.500000	20.000000	209.000000	148.365000	2.000000	40.625000	2023-04-05 00:00:00

50%	1145.000000	38.000000	216.000000	205.560000	3.000000	49.280000	2023-07-02 00:00:00

75%	1222.500000	60.000000	223.500000	258.590000	4.000000	61.655000	2023-09-27 12:00:00

max	1297.000000	79.000000	9999.000000	6178.250000	5.000000	97.510000	2023-12-30 00:00:00

std	85.592014	23.429119	1100.690799	621.050432	1.401750	16.238753	NaN



status

Cancelled    20262.47

Complete     40061.33

Return        5875.34

Unknown        707.61



## Reflection Questions

- Which data issue had the biggest impact on results?
  the revenue outliers
- What assumptions did you make during cleaning?
  all issues get deleted
- What risks exist if data validation is skipped?
  the outliers will get included in statistics and make a HUGE difference
